1. Instalacja bibliotek

In [25]:
%pip install requests psycopg2-binary


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: /usr/local/bin/python3.14 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


2. Import bibliotek

In [26]:
import requests
import psycopg2
from psycopg2 import sql
from contextlib import contextmanager

3. Konfiguracja bazy danych

In [27]:
DB_CONFIG = {
    "dbname": "postgres",
    "user": "postgres",
    "password": "admin",
    "host": "localhost",
    "port": "5432"
}

4. Pobieranie danych z API

In [28]:
def fetch_random_users(count=30):
    url = f"https://randomuser.me/api/?results={count}"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()['results']

5. Zarządzanie połączeniem (Context Manager)

In [29]:
@contextmanager
def get_db_cursor():
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    try:
        yield cur
        conn.commit()
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        cur.close()
        conn.close()

6. Inicjalizacja bazy danych

In [30]:
def init_database(cursor):
    cursor.execute("DROP TABLE IF EXISTS random_users;")
    cursor.execute("""
        CREATE TABLE random_users (
            id SERIAL PRIMARY KEY,
            first_name VARCHAR(100),
            last_name VARCHAR(100),
            email VARCHAR(150),
            address TEXT,
            country VARCHAR(100),
            age INT,
            gender VARCHAR(10)
        );
    """)

7. Zapis danych

In [31]:
def save_users_to_db(cursor, users):
    insert_stmt = """
        INSERT INTO random_users (first_name, last_name, email, address, country, age, gender)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """

    for u in users:
        # Logika transformacji danych oddzielona od zapytania SQL
        full_address = f"{u['location']['street']['name']} {u['location']['street']['number']}, {u['location']['city']}"
        data = (
            u['name']['first'],
            u['name']['last'],
            u['email'],
            full_address,
            u['location']['country'],
            u['dob']['age'],
            u['gender']
        )
        cursor.execute(insert_stmt, data)

8. Uruchomienie sekwencji zapisu danych z API do bazy danych PostgreSQL

In [32]:
def main():
    try:
        users = fetch_random_users(30)

        with get_db_cursor() as cur:
            init_database(cur)
            save_users_to_db(cur, users)
        print("Dane zapisane pomyślnie")
    except Exception as e:
        print(f"Wystąpił błąd: {e}")

if __name__ == "__main__":
    main()

Dane zapisane pomyślnie


9. Kwerendy

In [33]:
def run_queries():
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        cur = conn.cursor()

        # Zapytanie 1: Liczba mężczyzn
        cur.execute("SELECT COUNT(*) FROM random_users WHERE gender = 'male';")
        males = cur.fetchone()[0]
        print(f"1. Liczba mężczyzn: {males}")

        # Zapytanie 2: Średni wiek
        cur.execute("SELECT ROUND(AVG(age), 2) FROM random_users WHERE gender = 'male';")
        avg_age_male = cur.fetchone()[0]
        print(f"2. Średni wiek mężczyzn: {avg_age_male if avg_age_male else 0} lat")

        # Zapytanie 3: Liczba unikalnych krajów
        cur.execute("SELECT COUNT(DISTINCT country) FROM random_users;")
        countries = cur.fetchone()[0]
        print(f"3. Użytkownicy pochodzą z {countries} różnych krajów.")

    except Exception as e:
        print(f"Błąd podczas zapytań: {e}")
    finally:
        if 'conn' in locals():
            cur.close()
            conn.close()

run_queries()

1. Liczba mężczyzn: 20
2. Średni wiek mężczyzn: 57.55 lat
3. Użytkownicy pochodzą z 14 różnych krajów.
